# Gemma baseline, Qwen -> Gemma transfer, and one MLP for both models

All detectors: log-sum-exp pooling to 15 layers, no PCA, 4 x 128 MLP, 40 epochs, **standardization with each model's own
statistics** (source-train mean / std of that model; for transfer, the target model's own statistics).

1. **Gemma data:** Gemma-2-9B-it answers the six source datasets (`GEMMA_PER_SOURCE` each) and TriviaQA (`GEMMA_TRIVIAQA`), up to
   `MAX_NEW_TOKENS` = 64 new tokens (Qwen's saved data used 32); hidden states of the answer's last real token, BLEURT-20 labels.
   The questions are a seeded subset of the ones Qwen answered.
2. **A. Qwen detector** on Qwen data (reference).
3. **B. Gemma detector** on Gemma data: the Gemma baseline.
4. **C. Transfer:** Qwen detector on Gemma's TriviaQA answers, Gemma standardized with its own statistics (and the reverse).
5. **D. One MLP for both models:** trained on Qwen + Gemma sources (each model's own train / val / test split), tested on each
   model's TriviaQA.

**Inputs:** the features dataset `sala_qwen25_g14_6src_triviaqa_v1` and Gemma (Models -> google / gemma-2 / Transformers /
gemma-2-9b-it). GPU T4 x2, internet on (the question datasets are downloaded). Expect about 3 hours, mostly Gemma generation.

In [ ]:
# Clones the repo. Public repo: no credentials needed. If the anonymous clone fails (for example the repo is private
# again) it retries with the GITHUB_TOKEN Kaggle secret, which is passed to git only through an environment header,
# never in the command line or URL, so it cannot show up in tracebacks, logs or the saved clone.
import base64, os, subprocess, urllib.request

REPO, REF = "brainardphilemon/hallucination-probes", "main"   # pin a commit hash for reproducible runs
DEST = "/kaggle/working/hallucination-probes"
URL = f"https://github.com/{REPO}.git"
subprocess.run(["rm", "-rf", DEST], check=True)
clone = subprocess.run(["git", "clone", "-q", URL, DEST], env={**os.environ, "GIT_TERMINAL_PROMPT": "0"},
                       capture_output=True, text=True)
if clone.returncode:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError("Anonymous git clone failed and no GITHUB_TOKEN secret is attached "
                           "(Add-ons -> Secrets). git said: " + clone.stderr.strip()[-300:]) from None
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
           "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader", "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
    subprocess.run(["rm", "-rf", DEST], check=True)
    clone = subprocess.run(["git", "clone", "-q", URL, DEST], env=env, capture_output=True, text=True)
    if clone.returncode:
        try:  # 401 = token invalid/expired; 404 = token cannot see this repo (wrong repository or missing Contents: read)
            request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
            status = urllib.request.urlopen(request).status
        except Exception as exc:
            status = getattr(exc, "code", type(exc).__name__)
        raise RuntimeError(f"git clone failed; GitHub API answered {status} for {REPO} with your token. 401: token invalid or "
                           "expired. 404: the token has no read access to this repository (create it with Resource owner "
                           "brainardphilemon, repository hallucination-probes, Contents: Read-only).") from None
    del token, auth, env
subprocess.run(["git", "-C", DEST, "checkout", "-q", REF], check=True)
print(subprocess.run(["git", "-C", DEST, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r /kaggle/working/hallucination-probes/requirements-kaggle.txt
!pip install -q --no-deps -e /kaggle/working/hallucination-probes

In [ ]:
# ---- config -------------------------------------------------------------------------------
FEATURES_NAME = "sala_qwen25_g14_6src_triviaqa_v1"
SOURCES = ["tqa", "nq_open", "sciq", "popqa", "web_questions", "hotpotqa"]
GEMMA_PER_SOURCE = 500        # Gemma questions per source dataset (seeded subset)
GEMMA_TRIVIAQA = 1000         # Gemma TriviaQA test questions (seeded subset)
MAX_NEW_TOKENS = 64           # answer length limit (was 32)
MAX_INPUT_TOKENS = 1024       # prompt length limit (was 512)
GEMMA_ART = "/kaggle/working/gemma_art"
COMMON = "--pooling logsumexp --pool-layers 15 --standardize --hidden-dim 128 --hidden-layers 4 --epochs 40 --seed 42 --device cuda:0"

In [ ]:
import json
from pathlib import Path

def find_one(name, test, attach=""):
    root = Path("/kaggle/input")
    hits = [p for p in root.rglob(name) if test(p)]
    if len(hits) != 1:
        attached = sorted({str(p.relative_to(root)) for pattern in ("*", "*/*", "*/*/*") for p in root.glob(pattern)})
        raise RuntimeError(f"Expected exactly one {name} under /kaggle/input, found {hits}. {attach}\n"
                           "Attached right now:\n  " + "\n  ".join(attached[:60] or ["(nothing)"]))
    return hits[0]

def is_gemma_9b(config_path):
    c = json.loads(config_path.read_text())
    return c.get("model_type") == "gemma2" and c.get("hidden_size") == 3584 and c.get("num_hidden_layers") == 42

FEATURES = find_one(FEATURES_NAME, lambda p: (p / "generation").is_dir(),
                    "Add Input -> Datasets -> sala-g14-6-source-triviaqa-version-6-artifacts (nvthaai).")
GEMMA = find_one("config.json", is_gemma_9b,
                 "Add Input -> Models -> google / gemma-2 / Transformers / gemma-2-9b-it.").parent
print(FEATURES, GEMMA, sep="\n")

In [ ]:
# 1. Gemma answers, hidden states and BLEURT labels (resumable: re-running skips finished datasets).
GEN = f'--model "{GEMMA}" --model-revision "" --max-new-tokens {MAX_NEW_TOKENS} --max-input-tokens {MAX_INPUT_TOKENS} --device cuda:0 --bleurt-device cuda:0'
!hprobes extract --artifact-dir {GEMMA_ART} --domains {" ".join(SOURCES)} --samples-per-domain {GEMMA_PER_SOURCE} {GEN}
!hprobes extract --artifact-dir {GEMMA_ART} --domains triviaqa --samples-per-domain {GEMMA_TRIVIAQA} {GEN}

In [ ]:
# Answer lengths: how many Gemma answers still hit the limit
import json
from pathlib import Path
for d in [*SOURCES, "triviaqa"]:
    rows = [json.loads(l) for l in (Path(GEMMA_ART) / "generation" / d / "rows.jsonl").read_text().splitlines()]
    hit = sum(r["output_length"] >= MAX_NEW_TOKENS for r in rows)
    print(f"{d:14} {len(rows):5d} answers, {hit:4d} hit {MAX_NEW_TOKENS} tokens")

In [ ]:
# A. Qwen detector (reference)
!hprobes train --method mlp --artifact-dir "{FEATURES}" --output-dir /kaggle/working/qwen_std --sources {" ".join(SOURCES)} --targets triviaqa {COMMON}

In [ ]:
# B. Gemma detector: the Gemma baseline
!hprobes train --method mlp --artifact-dir {GEMMA_ART} --output-dir /kaggle/working/gemma_std --sources {" ".join(SOURCES)} --targets triviaqa {COMMON}

In [ ]:
# D. One MLP for both models (each model standardized with its own source-train statistics)
JOINT_SOURCES = " ".join([f"qwen:{d}" for d in SOURCES] + [f"gemma:{d}" for d in SOURCES])
!hprobes train --method mlp --artifact-dir qwen="{FEATURES}" gemma={GEMMA_ART} --output-dir /kaggle/working/joint_std --sources {JOINT_SOURCES} --targets qwen:triviaqa gemma:triviaqa {COMMON}

In [ ]:
# C. Transfer with the target model's own statistics, plus the summary table.
import sys
import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
sys.path.insert(0, "/kaggle/working/hallucination-probes/src")   # package was installed after the kernel started
from hprobes.data import load_labels
from hprobes.predict import Detector

def data(art, domain):
    rows, y, _ = load_labels(Path(art), domain, 0.5)
    return [str(r["id"]) for r in rows], y, np.load(Path(art) / "generation" / domain / "features.npy", mmap_mode="r")

def score(det, states, y, own_stats):
    out = det.predict(np.asarray(states), standardize_on_input=own_stats)
    return roc_auc_score(y, out["probability_correct"]), balanced_accuracy_score(y, out["predicted_label"])

q_ids, q_y, q_x = data(FEATURES, "triviaqa")
g_ids, g_y, g_x = data(GEMMA_ART, "triviaqa")
same = np.array([q_ids.index(i) for i in g_ids])           # Qwen rows for Gemma's TriviaQA questions
qdet, gdet = Detector.load("/kaggle/working/qwen_std/model.pt", "cuda"), Detector.load("/kaggle/working/gemma_std/model.pt", "cuda")
summ = {k: json.loads(Path(f"/kaggle/working/{k}/summary.json").read_text()) for k in ("qwen_std", "gemma_std", "joint_std")}
m = lambda s, t: (s["target_metrics"][t]["auroc"], s["target_metrics"][t]["balanced_accuracy"])
rows = [
    ("A  Qwen detector -> Qwen TriviaQA (all 9960)", *m(summ["qwen_std"], "triviaqa")),
    ("A  Qwen detector -> Qwen, same questions as Gemma", *score(qdet, q_x[same], q_y[same], False)),
    ("B  Gemma detector -> Gemma TriviaQA (baseline)", *m(summ["gemma_std"], "triviaqa")),
    ("C  Qwen detector -> Gemma TriviaQA, Gemma's own stats", *score(qdet, g_x, g_y, True)),
    ("C' Gemma detector -> Qwen, same questions, Qwen's own stats", *score(gdet, q_x[same], q_y[same], True)),
    ("D  One MLP -> Qwen TriviaQA (all)", *m(summ["joint_std"], "qwen:triviaqa")),
    ("D  One MLP -> Gemma TriviaQA", *m(summ["joint_std"], "gemma:triviaqa")),
]
table = pd.DataFrame(rows, columns=["detector -> test set", "AUROC", "balanced acc"]).round(4)
print(f"Gemma TriviaQA: {len(g_y)} answers, {int(g_y.sum())} BLEURT-correct; Qwen on the same questions: {int(q_y[same].sum())} correct")
display(table)
table.to_csv("/kaggle/working/summary_table.csv", index=False)